In [2]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import joblib
from sklearn.preprocessing import OneHotEncoder

##### Loading of dataset

In [3]:
df = pd.read_csv("../data/processed_barcode_data.csv")

##### Seperate Features and Target

In [4]:
X = df.drop(columns=["verification_status"])
y = df["verification_status"]

In [5]:
# To clear the Characters in my dataset
X.columns = (
    X.columns
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("<", "", regex=False)
    .str.replace(">", "", regex=False)
)

##### Perform Train-Test Split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

#### XGBoost

In [7]:
# Since the dataset is imbalanced(80:20), XGBoost has a parameter called scale_pos_weight to deal with the data imbalance.

negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print(scale_pos_weight)

3.9999545351216184


##### Building the Model

In [8]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="binary:logistic",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss"
)

In [9]:
# to check the column with Special Character
for col in X.columns:
    if any(char in col for char in ['[', ']', '<', '>']):
        print(col)

In [10]:
# Train the XGBoost Model 
xgb_model.fit(X_train, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [11]:
# Make Prediction

y_pred_xgb = xgb_model.predict(X_test)

In [12]:
# EVALUATION
print("Accuracy :", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall   :", recall_score(y_test, y_pred_xgb))
print("F1 Score :", f1_score(y_test, y_pred_xgb))

Accuracy : 0.5327344147814068
Precision: 0.2003751427173381
Recall   : 0.44680851063829785
F1 Score : 0.2766736107201171


In [13]:
# Classificaton Report
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.80      0.55      0.65     21995
           1       0.20      0.45      0.28      5499

    accuracy                           0.53     27494
   macro avg       0.50      0.50      0.47     27494
weighted avg       0.68      0.53      0.58     27494



In [14]:
# Confusion Matrix

cm_xgb = confusion_matrix(y_test, y_pred_xgb)

print(cm_xgb)

[[12190  9805]
 [ 3042  2457]]


##### Feature Importance

In [15]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.head(20)

,Feature,Importance
68,"dosage_form_INJECTION, SOLUTION, CONCENTRATE",0.023934
22,"dosage_form_CAPSULE, LIQUID FILLED",0.023211
81,dosage_form_LOTION,0.019657
120,dosage_form_SUPPOSITORY,0.016726
34,dosage_form_EMULSION,0.016594
14,dosage_form_CAPSULE,0.015994
116,"dosage_form_SPRAY, METERED",0.015844
87,dosage_form_OINTMENT,0.015361
301,route_'INTRAVENOUS',0.014647
122,"dosage_form_SUSPENSION, EXTENDED RELEASE",0.014409


#### Hyper parameter Tunning

In [16]:
xgb = XGBClassifier(
    objective="binary:logistic",
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight
)

In [17]:
# Defining the Hyperparameter Grid

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3, 0.5]
}

In [18]:
# Using Randomized Search
# Since the dataset is large, i had to use 20 random parameter combinations and 5-fold cross-validation
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=20,
    scoring="recall",
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)
# Using  recall for Scoring metrics because it is better to encourage to prioritize detecting fake drugs

In [19]:
# Train the Search

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.7, 0.8, ...], 'gamma': [0, 0.1, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'recall'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichretur

In [20]:
print("Best Parameters:")
print(random_search.best_params_)

Best Parameters:
{'subsample': 0.9, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0.5, 'colsample_bytree': 0.8}


In [21]:
print("Best Cross Validation Score:")
print(random_search.best_score_)

Best Cross Validation Score:
0.5087974539668105


In [22]:
best_xgb = random_search.best_estimator_

In [23]:
y_pred_best = best_xgb.predict(X_test)

In [24]:
# Evaluation 

print("Accuracy :", accuracy_score(y_test, y_pred_best))
print("Precision:", precision_score(y_test, y_pred_best))
print("Recall   :", recall_score(y_test, y_pred_best))
print("F1 Score :", f1_score(y_test, y_pred_best))

Accuracy : 0.5159307485269513
Precision: 0.20149824185904297
Recall   : 0.47935988361520276
F1 Score : 0.2837306926430224


In [25]:
print(classification_report(y_test, y_pred_best))

              precision    recall  f1-score   support

           0       0.80      0.53      0.63     21995
           1       0.20      0.48      0.28      5499

    accuracy                           0.52     27494
   macro avg       0.50      0.50      0.46     27494
weighted avg       0.68      0.52      0.56     27494



In [26]:
cm = confusion_matrix(y_test, y_pred_best)

print(cm)

[[11549 10446]
 [ 2863  2636]]


#### Comparison of All Models

So far, five machine learning models were evaluated using Accuracy, Precision, Recall, and F1-score. The performance of each model is presented and hyperparameter tuning produced the best model among all.

| Model                        |   Accuracy |  Precision |     Recall |   F1-Score |
| :--------------------------- | ---------: | ---------: | ---------: | ---------: |
| Random Forest (RF Baseline)  | **74.77%** |     19.94% |      8.67% |     12.09% |
| Random Forest (Class Weight) |     63.12% |     19.68% |     27.39% |     22.90% |
| Random Forest (SMOTE)        |     67.08% |     19.34% |     20.39% |     19.85% |
| XGBoost                      |     53.27% |     20.04% |     44.68% |     27.67% |
| **Tuned XGBoost**            | **51.59%** | **20.15%** | **47.94%** | **28.37%** |


#### Comparison Confusion Matrix 

| Model             | True Positives (Detected Fake) | False Negatives (Missed Fake) |
| ----------------- | -----------------------------: | ----------------------------: |
| Baseline RF       |                            477 |                         5,022 |
| RF + Class Weight |                          1,506 |                         3,993 |
| RF + SMOTE        |                          1,121 |                         4,378 |
| XGBoost           |                          2,457 |                         3,042 |
| **Tuned XGBoost** |                      **2,636** |                   **2,863**   |


Hyperparameter tuning further improved the XGBoost model, increasing the recall from 44.68% to 47.94% and the F1-score from 27.67% to 28.37%. The tuned model correctly identified 2,636 fake drugs, the highest among all the evaluated models. Although the overall accuracy decreased slightly, the improved detection capability makes the tuned XGBoost model the most suitable choice for the fake drug checker system.

#### Final Statement

After evaluating multiple machine learning models, including Random Forest (Baseline), Random Forest with Class Weight, Random Forest with SMOTE, XGBoost, and a hyperparameter-tuned XGBoost model, the tuned XGBoost model was selected as the final model. It achieved the highest F1-score (28.37%) and the highest recall (47.94%), indicating the best overall balance between correctly identifying fake drugs and minimizing missed detections. Therefore, it was chosen for deployment in the Streamlit-based fake drug barcode checker system

### Preparing for Deployment
 
Saving the Final Model (tuned XGBoost model)

In [ ]:
# Save trained model
joblib.dump(
    best_xgb,
    "../models/drug_model.pkl"
)

# Save feature names
joblib.dump(
    X.columns.tolist(),
    "../models/feature_columns.pkl"
)

print("Model and feature columns saved successfully!")

NameError: name 'best_xgb' is not defined

Verifying if the they are saved

In [38]:
model = joblib.load("./models/drug_model.pkl")
features = joblib.load("./models/feature_columns.pkl")

print("Model loaded successfully")
print("Number of features:", len(features))

Model loaded successfully
Number of features: 353
